# Short notes on executing Linux, GROMACS, and Python within Jupyter notebook

- Python is a natural environment of Jupyter. Type Python commands as they are.
- To enforce GROMACS and most Linux commands, you should add `!` in front of `gmx` or any other Linux command.
- There are few exception, *e.g.*, `cd`. You should use `%` instead of `!`.

In [ ]:
# general imports 
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video

# Why GROMACS?

- Extremely high performance compared to all other molecular dynamics engines.
- User-friendly, all necessary files are written in clear text format.
- No programming involved to use it straight away.
- Huge community of users, forums, tutorials, excellent documentation.
- Both input files and trajectories are independent of hardware used to generate them.
- Compact way of storing trajectory data (.xtc format) - can be read by other tools and libraries for visualization and advanced analyses.
- Large selection of inbox tools for system building (including building proteins from Protein Data Bank) and routine analyses.
- High parallelization capabilities and excellent GPU-based acceleration.
- GROMACS is free software available under the GNU Lesser General Public License.

## GROMACS tutorials

- https://tutorials.gromacs.org/ (official ones)
- http://www.mdtutorials.com/gmx/ (unofficial ones but those were done probably by every person working in GROMACS)

# How to use?

Every GROMACS command starts with <code>gmx</code> prefix. For instance, today we will use following commands: <code>gmx solvate</code>, <code>gmx editconf</code>, <code>gmx grompp</code>, <code>gmx mdrun</code>, and <code>gmx energy</code>.

Are there are many <code>gmx</code> commands? Yes! Let's have a look at the full list:

In [ ]:
# !gmx help commands

### Also available here 

- https://manual.gromacs.org/current/user-guide/cmdline.html

# Description of GROMACS commands

If you are interested in details of one particular command, just type <code>gmx help {name_of_the_command}</code> or <code>gmx {name_of_the_command} -h</code>

In [ ]:
!gmx help solvate
# !gmx solvate -h

# Hierarchy of GROMACS files

- `.gro` &mdash; contains atomic coordinates, velocities (optionally), and box dimensions.

- `.top` &mdash; provides the molecular topology information which describes the structure of molecules, the type of atoms, bonds, angles, dihedrals, and non-bonded parameters. It contains the definitions of atom types, bond types, and force field parameters. It can include other topology fragments (often saved with the `.itp` extension) to allow for modularity.

- `.itp` &mdash; individual topology fragments that can be included in the main `.top` file.
 
- `.mdp` &mdash; specifies the parameters for running a simulation. We will discuss `.mdp` parameters throughout the course.

- `.tpr` &mdash; a compiled binary file containing all the information from the files above needed to start a simulation.

# Today's Practical

We will prepare a set of water/methanol mixtures. This is an example system, which will introduce you to GROMACS, its basic commands and usage, and principal steps of performing molecular simulations.

This exersise is also designed to detail concepts like simulation time, equilibration, and basic analysis necessary to ensure the realibilty of your simulations.

## What files do we have?

In [ ]:
!ls -l

## Short folders' decsription

### force_field

Contains force field ($V(r)$ in the lecture) files necessary to execute our simulations of water/methanol mixtures.

### images

Example images.

### scripts

Contains `.mdp` files we will use today.

### solvents

Starting atomic configurations for water and methanol.

### topologies

`.itp` topologies of water and methanol.

### videos

Example videos.

### week2.ipynb

Jupyter notebook we are using right now.

## Create a folder where we will be working today

In [ ]:
!mkdir mixtures/

In [ ]:
%cd mixtures/

In [ ]:
!ls -l

## Solvents

In [ ]:
!head ../solvents/water_four_site.gro
print('...')
!tail ../solvents/water_four_site.gro

### Four-site model of water

- One additional virtual (ghost) atom on top of an oxygen and two hydrogens
    - necessary to better reproduce hydrogen bonding and dipole moment
    - the virtual atom also represents the lone pairs on the oxygen atom
- Common for water models in order to better reproduce experimental properties
- In biosimulations, four-site water models are used less than three-site (historical reasons mainly)

![water](../images/water_four_site_noMW.png)
![water](../images/water_four_site.png)

In [ ]:
!head ../solvents/methanol_three_site.gro
print('...')
!tail ../solvents/methanol_three_site.gro

### Three-site model of methanol

- Methyl group modeled as a single sphere (essentially, carbon)
    - common for modeling alcohols
- Better to reproduce experimental properties, more computationally efficient

![methanol](../images/methanol_three_site.png)

## How will we prepare a mixture?

- Prepare a box of water.
- Prepare a box of methanol.
- Mix them!

### Procedure

- Boxes of water with 1000, 750, 500, and 250 water molecules.
- Boxes of methanol with 1000, 750, 500, and 250 methanol molecules.
- Five final systems:
    - Pure water (1000 molecules)
    - Pure methanol (1000 molecules)
    - Mixture 3 to 1 (750 waters and 250 methanols)
    - **Mixture 1 to 1 (500 waters and 500 methanols) (we will focus on this one today)**
    - Mixture 1 to 3 (250 waters and 750 methanols)

### Using `gmx solvate` to prepare initial boxes

- `gmx solvate` command is perfectly suited for our task
- `-cs` &mdash; with that we will fill the box? First, we will prepare boxes with water.
- `-box` &mdash; dimensions/size of the box
- `-maxsol` &mdash; control number of solvent molecules; if more molecules fit into the box than we need
- `-o` &mdash; name of the file we will generate

In [ ]:
!gmx solvate -cs ../solvents/water_four_site.gro -box 3.2 3.2 3.2 -maxsol 1000 -o w1000.gro
!gmx solvate -cs ../solvents/water_four_site.gro -box 2.9 2.9 2.9 -maxsol 750 -o w750.gro 
!gmx solvate -cs ../solvents/water_four_site.gro -box 2.5 2.5 2.5 -maxsol 500 -o w500.gro 
!gmx solvate -cs ../solvents/water_four_site.gro -box 2.1 2.1 2.1 -maxsol 250 -o w250.gro 

In [ ]:
!ls -l

In [ ]:
!gmx solvate -cs ../solvents/methanol_three_site.gro -box 4.2 4.2 4.2 -maxsol 1000 -o m1000.gro
!gmx solvate -cs ../solvents/methanol_three_site.gro -box 3.8 3.8 3.8 -maxsol 750 -o m750.gro 
!gmx solvate -cs ../solvents/methanol_three_site.gro -box 3.3 3.3 3.3 -maxsol 500 -o m500.gro 
!gmx solvate -cs ../solvents/methanol_three_site.gro -box 2.8 2.8 2.8 -maxsol 250 -o m250.gro 

In [ ]:
!ls -l

### Renaming files for clarity

In [ ]:
!mv w1000.gro pure_water.gro
!mv m1000.gro pure_methanol.gro
!ls -l

<img src='../images/w500.png' width=600 height=450 alt='water_box'>
<img src='../images/m500.png' width=600 height=450 alt='methanol_box'>

## Merging boxes into one

- Shift the atomic coordinates in one of them, then combine files, and change the box size.
- We shift the coordinates to avoid clashes when two boxes are combined

### `gmx editconf`

- `-f` &mdash; input file
- `-o` &mdash; output file
- `-translate` &mdash; changes the coordinates in $x$, $y$, and $z$ dimensions

We will shift the atomic coordinates in boxes with methanol.

In [ ]:
!gmx editconf -f m750.gro -o m750.gro -translate 2.3 2.3 2.3
!gmx editconf -f m500.gro -o m500.gro -translate 2.7 2.7 2.7
!gmx editconf -f m250.gro -o m250.gro -translate 3.1 3.1 3.1

In [ ]:
!ls -l

### Overwriting the files

- If GROMACS wants to write a file called `m500.gro` and such already exists, it renames the existing one to `#m500.gro.1#` before writing a new one.
- Useful to back track the data.
- Can be deleted if you are sure you do not need them.

In [ ]:
!rm -f *#*

In [ ]:
!head w500.gro
print('...')
!tail w500.gro
print('......')
!head m500.gro
print('...')
!tail m500.gro

## Merging structures into one using Linux tools

- Take all lines of the water box except the last one
- Take all lines of the methanol box except the first two
- Update the total number of atoms on the second line
- Update the box size

In [ ]:
!{ head -n -1 w750.gro; tail -n +3 m250.gro; } | awk 'NR==2 {print "3750"}; NR!=2' > mix_w3_m1.gro
!{ head -n -1 w500.gro; tail -n +3 m500.gro; } | awk 'NR==2 {print "3500"}; NR!=2' > mix_w1_m1.gro
!{ head -n -1 w250.gro; tail -n +3 m750.gro; } | awk 'NR==2 {print "3250"}; NR!=2' > mix_w1_m3.gro

In [ ]:
!ls -l

### Update the box size
- New box sizes are chosen to have both solvents within the box boundaries
- `-resnr 1` makes cosmetic yet useful change to a new `.gro` file correcting the numbering of molecules
- `gmx editconf` usually automatically applies centering (which is a keyword `-c`) whenever changes to the box are made. We do not need this here, therefore, we use `-noc` option

In [ ]:
!gmx editconf -f mix_w3_m1.gro -o mix_w3_m1.gro -box 6.3 6.3 6.3 -resnr 1 -noc
!gmx editconf -f mix_w1_m1.gro -o mix_w1_m1.gro -box 6.2 6.2 6.2 -resnr 1 -noc
!gmx editconf -f mix_w1_m3.gro -o mix_w1_m3.gro -box 6.1 6.1 6.1 -resnr 1 -noc

<img src='../images/mix_w1_m1_img1.png' width=600 height=450 alt='mixture_w1_m1_img1'>
<img src='../images/mix_w1_m1_img2.png' width=600 height=450 alt='mix_w1_m1_img2'>

### Topology `.top` file

In [ ]:
topology = """; force field parameters
#include "../force_field/forcefield.itp"
#include "../topologies/tip4p_2005.itp"
#include "../topologies/methanol_three_site.itp"

[ system ]
; name of the system
mixture w1:m1

[ molecules ]
; compound   #mols
SOL          500
MET          500
"""

with open('mix_w1_m1.top', 'w') as outFile:
    outFile.write(topology)

In [ ]:
!cat ../force_field/forcefield.itp

In [ ]:
!cat ../force_field/ffnonbonded.itp

In [ ]:
!cat ../topologies/tip4p_2005.itp

In [ ]:
!cat ../topologies/methanol_three_site.itp

## Running simulations

- Two key commands &mdash; `gmx grompp` and `gmx mdrun`

### `gmx grompp`

- `-f` &mdash; your `.mdp` file that contains a simulation protocol
- `-p` &mdash; your `.top` file
- `-c` &mdash; your `.gro` file
- `-o` &mdash; resulting binary `.tpr` file
- `-maxwarn` &mdash; sometimes GROMACS gives warning that does not allow you to generate a `.tpr`. You can specify max number of warnings allowed. Be careful.
- `-po` &mdash; output text file with used simulation parameters; similar to the input `-f` `.mdp` file but also includes default, not explicitly mentioned options

### `gmx mdrun`

- `-deffnm` &mdash; output files will be called the same as input `.tpr`
- `-v` &mdash; show a live progress of your simulations (we will not use it in some cases today to not overload the Jupyter notebook)
- `-quiet` &mdash; suppress most of the verbose output when executing a command (keep in mind: sometimes better not to suppress them)

<div class="alert alert-block alert-warning">
Today we will perform various types of simulations. More about them from the theoretical standpoint &mdash; next week.
</div>

## Energy minimization

- Required to ensure that the built system has no steric overlaps/clashes and inappropriate geometries.
- The idea of the algorithm we will use (steepest descent) is to move atoms a bit until there are no large forces.

### `.mdp` options

- https://manual.gromacs.org/current/user-guide/mdp-options.html (full list and description)

In [ ]:
!cat ../scripts/min.mdp

### Important pieces for today

In [ ]:
!gmx grompp -f ../scripts/min.mdp -p mix_w1_m1.top -c mix_w1_m1.gro -o min_mix_w1_m1.tpr -po min_mdout_mix_w1_m1.mdp

In [ ]:
!gmx mdrun -deffnm min_mix_w1_m1

### Minimization/equilibration can generate stepXXX.pdb files 

- Generated to investigate possible issues with the system if any.
- Quite common particularly for minimization.
- Can be deleted if you do not have any troubles at the end.

In [ ]:
!rm -f *step*

# Hierarchy of output simulation files

- `.gro` &mdash; contains **final** atomic coordinates, velocities (optionally), and box dimensions.

- `.xtc` &mdash; a compressed trajectory file containing the atomic coordinates at different time frames throughout the simulation. Due to its compression, it is smaller in size compared to other trajectory formats, but it does not store velocities or forces. **99% choice for biosimulations.**

- `.trr` &mdash; a full precision trajectory file. Unlike the `.xtc` file, it contains atomic coordinates, velocities, and forces without compression. This makes it larger in size, but it can be more useful for certain analysis tasks where precision is crucial. **Almost never used in biosimulations.**

- `.edr` &mdash; stores various energy terms (e.g., kinetic energy, potential energy, total energy, temperature, pressure) and other system information throughout the simulation.
 
- `.log` &mdash; a text file that provides a detailed record of the simulation progress, including settings, energy terms at each step, and any warnings or errors.

- `.cpt` &mdash; a checkpoint file that contains all necessary information to continue a simulation from where it left off.

## Analysing simulations

- Multiple built-in GROMACS tools are available
- There are some that used almost always

### `gmx energy`

-  Tool to work with a binary `.edr` energy file
- `-f` &mdash; input `.edr` file
- `-o` &mdash; output `.xvg` file; you can then plot it &mdash we will do it now.

In [ ]:
!echo 'Potential' | gmx energy -f min_mix_w1_m1.edr -o pot_min_mix_w1_m1.xvg

In [ ]:
!head -n 50 pot_min_mix_w1_m1.xvg

In [ ]:
potential = np.loadtxt('pot_min_mix_w1_m1.xvg', comments=['#', '@'])
plt.plot(potential[:, 0], potential[:, 1], 'r*')
plt.legend(['minimization'])
plt.xlabel('Time (ps)')
plt.ylabel('Potential Energy  (kJ/mol)')
plt.show()

### More about matplotlib

https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html

## Equilibration

- **Every** system first needs to be equilibrated. The more complex systems, the longer simulations times are required.
- For our system, we expect the equilibration to result in full mixing of water and methanol.

### Equilibration with constant volume (NVT ensemble, more about ensembles during Week 3)

In [ ]:
!cat ../scripts/eq_nvt.mdp

### Important pieces for today

In [ ]:
!gmx grompp -f ../scripts/eq_nvt.mdp -maxwarn 1 -p mix_w1_m1.top -c min_mix_w1_m1.gro -o eqnvt_mix_w1_m1.tpr -po eqnvt_mdout_mix_w1_m1.mdp

In [ ]:
# !gmx mdrun -deffnm eqnvt_mix_w1_m1 -v
# !bash ../../wolf/run_md_wolf31.sh eqnvt_mix_w1_m1
!bash ../../wolf/run_md_wolf32.sh eqnvt_mix_w1_m1

## A few comments about running jobs at local wolf cluster

- `pnodes` &mdash; show computers (nodes) available
- `pjobs` &mdash; show your simulations/jobs
- `pinfo` &mdash; show info about simulation in the current directory (important later)

<img src='../images/eq_nvt_mixture_w1_m1_img1.png' width=600 height=450 alt='eq_nvt_mixture_w1_m1_img1'>
<img src='../images/eq_nvt_mixture_w1_m1_img2.png' width=600 height=450 alt='eq_nvt_mixture_w1_m1_img2'>

In [ ]:
!echo 'Temperature' | gmx energy -f eqnvt_mix_w1_m1.edr -o temp_eqnvt_mix_w1_m1.xvg

In [ ]:
!head -n 50 temp_eqnvt_mix_w1_m1.xvg

In [ ]:
temperature = np.loadtxt('temp_eqnvt_mix_w1_m1.xvg', comments=['#', '@'])
plt.plot(temperature[:, 0], temperature[:, 1], 'r*')
plt.axhline(y=298.15, color='b', linestyle='--')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Temperature (K)')
plt.show()

## TASK

Calculate the potential energy from `evnvt` simulation and plot it.

### Equilibration with constant pressure (NPT ensemble, more about ensembles during Week 3)

In [ ]:
!cat ../scripts/eq_npt.mdp

In [ ]:
!gmx grompp -f ../scripts/eq_npt.mdp -maxwarn 2 -p mix_w1_m1.top -c eqnvt_mix_w1_m1.gro -o eqnpt_mix_w1_m1.tpr -po eqnpt_mdout_mix_w1_m1.mdp

### Running a new simulation in the same directory (this is a feature of clusters at MUNI)

- Info files generated by a previous simulations should be deleted (can be done if you are) or moved somewhere (a safe option)

In [ ]:
!ls -l

In [ ]:
!mkdir eqnvt_wolf_files/

In [ ]:
!mv run.sh* eqnvt_wolf_files/

In [ ]:
# !gmx mdrun -deffnm eqnpt_mix_w1_m1 -v
# !bash ../../wolf/run_md_wolf31.sh eqnpt_mix_w1_m1
!bash ../../wolf/run_md_wolf32.sh eqnpt_mix_w1_m1

<img src='../images/eq_nvt_mixture_w1_m1_img1.png' width=600 height=450 alt='eq_nvt_mixture_w1_m1_img1'>
<img src='../images/eq_npt_mixture_w1_m1_img1.png' width=600 height=450 alt='eq_npt_mixture_w1_m1_img1'>
<img src='../images/eq_npt_mixture_w1_m1_img2.png' width=600 height=450 alt='eq_npt_mixture_w1_m1_img2'>

In [ ]:
Video('../videos/video_eq_npt.mp4', width=800, height=600, embed=True)

## Do we save every time step in a simulation trajectory?

- It is not necessary in the majority of simulations, especially biological ones.
- You can, of course, still do it, but then you generate terabytes of data.
- Usually, trajectories are saved every 1&ndash;100 ps depending on a system.

## Simulation with every time step being stored

In [ ]:
Video('../videos/video_every_time_step.mp4', width=800, height=600, embed=True)

In [ ]:
!echo 'Pressure' | gmx energy -f eqnpt_mix_w1_m1.edr -o pres_eqnpt_mix_w1_m1.xvg

In [ ]:
pressure = np.loadtxt('pres_eqnpt_mix_w1_m1.xvg', comments=['#', '@'])
plt.plot(pressure[:, 0], pressure[:, 1], 'r*')
plt.axhline(y=1, color='b', linestyle='--')
plt.legend(['equilibration'])
plt.xlabel('Time (ps)')
plt.ylabel('Pressure (bar)')
plt.show()

## TASK

Calculate the potential energy and temperature from `evnpt` simulation and plot it.

### Production run (in biosimulations, almost always NPT ensemble)

<div class="alert alert-block alert-warning">
Production simulations are where the main results coming from &mdash; this is what you publish
</div>

In [ ]:
!cat ../scripts/prod_md.mdp

In [ ]:
!gmx grompp -f ../scripts/prod_md.mdp -maxwarn 1 -p mix_w1_m1.top -c eqnpt_mix_w1_m1.gro -o prod_mix_w1_m1.tpr -po prod_mdout_mix_w1_m1.mdp

In [ ]:
!mkdir eqnpt_wolf_files/

In [ ]:
!mv run.sh* eqnpt_wolf_files/

In [ ]:
# !gmx mdrun -deffnm prod_mix_w1_m1 -v
# !bash ../../wolf/run_md_wolf31.sh prod_mix_w1_m1
!bash ../../wolf/run_md_wolf32.sh prod_mix_w1_m1

In [ ]:
!echo 'Density' | gmx energy -f prod_mix_w1_m1.edr -o dens_prod_mix_w1_m1.xvg

density = np.loadtxt('dens_prod_mix_w1_m1.xvg', comments=['#', '@'])
plt.plot(density[:, 0], density[:, 1], 'r*')
plt.legend(['production'])
plt.xlabel('Time (ps)')
plt.ylabel('Density (kg/$\mathrm{m^3}$)')
plt.show()

## TASK

Add the average to the graph.

## TASK

Calculate the potential energy, temperature, and pressure from `prod` simulation and plot it.

## Running long simulations

- We performed a short simulation on a small system.
- Long simulations of a big system usually performed using computers equipped with GPU, computer clusters, supercomputers, etc.
- Advantage: you start a simulation, work on something else, come back later, and the finished simulation is waiting for you.

## Submitting jobs at wolf cluster

We will generate a new `.tpr` for a long simulation.

In [ ]:
!cat ../scripts/long_md.mdp

In [ ]:
!gmx grompp -f ../scripts/long_md.mdp -maxwarn 1 -p mix_w1_m1.top -c prod_mix_w1_m1.gro -o long_mix_w1_m1.tpr -po long_mdout_mix_w1_m1.mdp

In [ ]:
!mkdir prod_wolf_files/

In [ ]:
!mv run.sh* prod_wolf_files/

In [ ]:
# !bash ../../wolf/run_md_wolf31.sh long_mix_w1_m1
!bash ../../wolf/run_md_wolf32.sh long_mix_w1_m1

... and now we just wait. :)

## ADDITIONAL TASK

Run simulations for another mixture of your choice. If you can do all of them, it will be great. :)